### Volume EM Development

In [1]:
%load_ext autoreload 
%autoreload 2

import os

import yaml
import logging
from fibsem import utils
from fibsem.structures import FibsemRectangle, BeamType, FibsemMillingSettings
from fibsem.milling.tasks import FibsemMillingTaskConfig, FibsemMillingStage, MillingAlignment
from fibsem.milling.patterning import RectanglePattern
from fibsem.applications.volume_em.structures import VolumeAcquisitionConfig, VolumeMillingConfig, VolumeEMConfig
from fibsem.applications.volume_em.tasks import VolumeEMTask, run_volume_em_task

microscope, settings = utils.setup_session()


Default configuration default-configuration. Configuration Path: C:\Users\rohit\Documents\Code\newFIBSEM\fibsem-os-NYSBC\fibsem\config\microscope-configuration.yaml
2026-03-04 09:50:10,222 — root — INFO — _setup_image_iterators:466 — SEM or FIB data path not configured in simulator settings, using random noise generation
2026-03-04 09:50:10,227 — root — INFO — connect_to_microscope:282 — Microscope client connected to DemoMicroscope with serial number 123456 and software version 0.1
2026-03-04 09:50:10,247 — root — INFO — setup_session:325 — Finished setup for session: demo_2026-03-04-09-50-10AM


In [3]:

acquisitions = {
    "Acquisition 01": VolumeAcquisitionConfig(name="Acquisition 01", 
                                              state=microscope.get_microscope_state(BeamType.ELECTRON)),
    "Acquisition 02": VolumeAcquisitionConfig(name="Acquisition 02", 
                                              state=microscope.get_microscope_state(BeamType.ELECTRON)),
    "Acquisition 03": VolumeAcquisitionConfig(name="Acquisition 03", 
                                              state=microscope.get_microscope_state(BeamType.ELECTRON))
}

acquisitions["Acquisition 02"].image.reduced_area = FibsemRectangle(left=0.25, top=0.25, width=0.5, height=0.5)
acquisitions["Acquisition 03"].image.hfw = 500e-6
acquisitions["Acquisition 03"].image.resolution = (6144, 4096)


milling_task_config = FibsemMillingTaskConfig(
    name = "Volume EM Milling Task",
    field_of_view=400e-6,
    alignment=MillingAlignment(enabled=True),
    stages=[
        FibsemMillingStage(name="Volume EM Stage",
                           pattern=RectanglePattern(width=200e-6, height=1e-6, depth=5e-6),
                           milling=FibsemMillingSettings(milling_current=1e-9, milling_voltage=30000)
    )
    ],
)

milling_config = VolumeMillingConfig(
    name="Volume EM Milling Config",
    n_steps=100,
    step_size=100e-9,
    current_step=0,
    state=microscope.get_microscope_state(BeamType.ION),
    config=milling_task_config
)
# milling_config.config.acquisition.enabled = False

volume_config = VolumeEMConfig(
    name="Volume EM Config",
    description="Volume EM acquisition and milling configuration",
    path=os.path.join(os.getcwd(), "test-volume-em-experiment"),
    milling=milling_config,
    acquisitions=acquisitions
)


# QUERY:
# multiple milling tasks?
# other material removal tasks?
# gui for displaying milling steps should be different than regular milling gui?


In [7]:
print(volume_config.path)
filename = volume_config.save()

c:\Users\rohit\Documents\Code\newFIBSEM\fibsem-os-NYSBC\fibsem\applications\volume_em\test-volume-em-experiment


In [ ]:
run_volume_em_task(microscope, volume_config)

In [ ]:
filename = volume_config.save()

In [ ]:
from pprint import pprint

cfg = VolumeEMConfig.load(filename)
pprint(cfg.to_dict())

run_volume_em_task(microscope, cfg)